# Õppetund 11 - Agendi-agendi (A2A) protokoll


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on A2A-protokoll?

The **Agent-to-Agent (A2A) protokoll** on avatud standard, mis võimaldab tehisintellektil põhinevatel agentidel suhelda,
avastada üksteist ja teha koostööd — isegi kui need on ehitatud erinevatele raamistikutele või majutatud
erinevate teenuste poolt.

Peamised mõisted:

- **Avastamine** – Agendid avaldavad *Agendi kaart*, mis kirjeldab nende võimekust, muutes selle
  lihtsaks teiste agentide (või orkestreerijate) jaoks leida õiget spetsialisti ülesandeks.
- **Sõnumite edastamine** – Agendid vahetavad struktureeritud sõnumeid ühise protokolli kaudu, nii et a
  ühe agendi päringut saab teise poolt mõista ja täita sõltumata selle sisemisest rakendusest.
- **Tööülesande elutsükkel** – A2A määratleb seisundid nagu *esitatud*, *töös*, *täidetud* ja
  *ebaõnnestunud*, andes orkestreerijale täieliku ülevaate sellest, kuidas delegeeritud ülesanne edeneb.

Selles õppetükis simuleerime A2A-laadset koostööd, lisades töövoogu kolm spetsialiseerunud reisiagenti
— iga agent panustab oma ekspertiisi ja edastab tulemused järgmisele.


## Spetsialiseeritud reisiagentide loomine


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Mitmeagendi koostöö töövoo kaudu

Me ühendame need kolm agenti järjestikusesse töövoogu, mis peegeldab agentilt-agentile (A2A) sõnumivahetust:

1. **CurrencyExchangeAgent** võtab vastu kasutaja päringu ja annab valuutanõuandeid.
2. **ActivityPlannerAgent** saab rikastatud konteksti ja lisab tegevuste soovitusi.
3. **TravelManagerAgent** sünteesib mõlemad sisendid lõplikuks reisikokkuvõtteks.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A mõistmine tootmises

Tootmiskeskkonnas avab A2A protokoll võimsaid teenustevahelisi stsenaariume:

| Capability | Description |
|---|---|
| **Raamistikutevaheline koostalitlusvõime** | Ühe raamistikuga loodud agent võib delegeerida ülesandeid agendile, mis on loodud mõne muu A2A-ühilduva raamistikuga, võimaldades tõelist organisatsioonidevahelist koostalitlusvõimet. |
| **Teenusepiirid** | Agendid võivad paikneda eraldi mikroteenustes, pilveregioonides või isegi erinevates organisatsioonides, kuid töötada endiselt sujuvalt koos. |
| **Dünaamiline avastamine** | Orkestreerija saab käituse ajal pärida Agent Card registrit, et leida antud alamülesande jaoks kõige sobivam spetsialist. |
| **Voogedastus ja push-teavitused** | A2A toetab Server-Sent Events (SSE) reaalajas edusammude värskendusteks ja push-teavitusi pikaajaliste ülesannete jaoks. |

Ülaltoodud töövoog, mille ehitasime, on selle mustri lihtsustatud protsessisisene versioon. Tegelikus juurutuses avaldaks iga agent HTTP lõpp-punkti, avalikustaks Agent Cardi ja suhtleks A2A JSON-RPC protokolli kaudu.


## Kokkuvõte

Selles õppetunnis õppisite:

1. **Mis on A2A-protokoll** — avatud standard agentidevaheliseks avastamiseks, sõnumivahetuseks,
   ja ülesannete haldamiseks.
2. **Kuidas luua spetsialiseeritud agente** — valuutavahetuse agent, tegevuste planeerija agent,
   ja reisihalduri orkestreerija.
3. **Kuidas ühendada agendid töövoogu** — kasutades `WorkflowBuilder` järjestikuse
   sõnumiedastuse modelleerimiseks agentide vahel.
4. **Kuidas A2A töötab tootmises** — võimaldades raamistikudevahelist, teenustevahelist koostööd
   dünaamilise avastamise ja voogedastuse uuendustega.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Lahtiütlus:
See dokument on tõlgitud tehisintellektil põhineva tõlke-teenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüdleme täpsuse poole, pidage palun meeles, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle algkeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või väärtõlgenduste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
